In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

SORTIE = RACINE / "data" / "processed"

valeurs = pd.read_csv(SORTIE / "valeurs_portefeuilles.csv", index_col=0)
comparaisons = pd.read_csv(SORTIE / "valeurs_comparaisons.csv", index_col=0)
tout = valeurs.join(comparaisons)

print(valeurs.shape, comparaisons.shape)

(6709, 20) (6709, 5)


In [2]:
lignes = []
for serie in tout.columns:
    niveau = tout[serie].dropna()
    quotidien = niveau.pct_change().dropna()

    mensuel = niveau.groupby(pd.Series(niveau.index).str[:7].values).last()
    mensuel = mensuel.pct_change().dropna()

    lignes.append({"serie": serie,
                   "vol_quotidienne": quotidien.std() * np.sqrt(252),
                   "vol_mensuelle": mensuel.std() * np.sqrt(12),
                   "autocorrelation": quotidien.autocorr(1),
                   "obs_quotidiennes": len(quotidien),
                   "obs_mensuelles": len(mensuel)})

frequence = pd.DataFrame(lignes).set_index("serie")
frequence["rapport"] = frequence.vol_quotidienne / frequence.vol_mensuelle

print(frequence.round(4).to_string())
print()
print("rapport median %.3f | min %.3f | max %.3f"
      % (frequence.rapport.median(), frequence.rapport.min(), frequence.rapport.max()))

          vol_quotidienne  vol_mensuelle  autocorrelation  obs_quotidiennes  obs_mensuelles  rapport
serie                                                                                               
P1_reeq            0.2173         0.1910          -0.0446              6708             320   1.1375
P1_cons            0.2352         0.1997          -0.0569              6708             320   1.1777
P10_reeq           0.2302         0.1986          -0.0251              6708             320   1.1590
P10_cons           0.2722         0.2592          -0.0319              6708             320   1.0504
P2_reeq            0.2382         0.2146          -0.0298              6708             320   1.1097
P2_cons            0.2556         0.2212          -0.0446              6708             320   1.1552
P3_reeq            0.1846         0.1517          -0.0785              6708             320   1.2166
P3_cons            0.2300         0.2164          -0.0547              6708             320

In [3]:
FENETRES = [252, 756]

lignes = []
for serie in tout.columns:
    r = tout[serie].dropna().pct_change().dropna()
    ligne = {"serie": serie, "periode_complete": r.std() * np.sqrt(252)}
    for n in FENETRES:
        glissante = r.rolling(n).std() * np.sqrt(252)
        ligne[f"med_{n}"] = glissante.median()
        ligne[f"min_{n}"] = glissante.min()
        ligne[f"max_{n}"] = glissante.max()
        ligne[f"amplitude_{n}"] = glissante.max() / glissante.min()
    lignes.append(ligne)

fenetres = pd.DataFrame(lignes).set_index("serie")
print(fenetres[["periode_complete", "med_252", "min_252", "max_252", "amplitude_252"]]
      .round(4).to_string())
print()
print("erreur relative d'estimation : 252 jours %.2f %%, 756 jours %.2f %%"
      % (100 / np.sqrt(2 * 252), 100 / np.sqrt(2 * 756)))

          periode_complete  med_252  min_252  max_252  amplitude_252
serie                                                               
P1_reeq             0.2173   0.1753   0.0891   0.4737         5.3154
P1_cons             0.2352   0.1999   0.0961   0.4861         5.0606
P10_reeq            0.2302   0.1863   0.1078   0.5658         5.2489
P10_cons            0.2722   0.2093   0.1120   0.6069         5.4185
P2_reeq             0.2382   0.1944   0.1006   0.4949         4.9174
P2_cons             0.2556   0.2193   0.1082   0.4999         4.6198
P3_reeq             0.1846   0.1497   0.0787   0.4271         5.4272
P3_cons             0.2300   0.1813   0.0918   0.5026         5.4749
P4_reeq             0.3115   0.2309   0.1183   0.5973         5.0480
P4_cons             0.3436   0.2588   0.1459   0.6458         4.4255
P5_reeq             0.3218   0.2574   0.1444   0.6342         4.3916
P5_cons             0.3473   0.2826   0.1471   0.6311         4.2897
P6_reeq             0.1894   0.161

In [4]:
SEUIL_TENSION = 0.15
SEUIL_MAJEUR = 0.20
REFERENCE = "SPY"


def episodes_de_tension(niveau, seuil):
    sommet = niveau.cummax()
    groupe = (sommet != sommet.shift()).cumsum()
    lignes = []
    for _, segment in niveau.groupby(groupe):
        repli = segment / segment.iloc[0] - 1
        if repli.min() <= -seuil:
            dernier = segment.index[-1]
            retour = (niveau.index[niveau.index.get_loc(dernier) + 1]
                      if dernier != niveau.index[-1] else None)
            lignes.append({"debut": segment.index[0], "creux": repli.idxmin(),
                           "repli": repli.min(), "retour": retour,
                           "seances": len(segment)})
    return pd.DataFrame(lignes)


marche = tout[REFERENCE].dropna()
tensions = episodes_de_tension(marche, SEUIL_TENSION)
tensions["majeur"] = tensions.repli <= -SEUIL_MAJEUR

en_tension = pd.Series(False, index=tout.index)
for r in tensions.itertuples():
    en_tension.loc[r.debut:r.creux] = True

tensions.to_csv(SORTIE / "episodes_tension.csv", index=False, encoding="utf-8")

print(tensions.to_string(index=False, float_format=lambda x: f"{x:.1%}"))
print()
print("seances en tension : %d sur %d, soit %.1f %%"
      % (en_tension.sum(), len(en_tension), 100 * en_tension.mean()))
print("episodes majeurs :", int(tensions.majeur.sum()))

     debut      creux  repli     retour  seances  majeur
2000-03-24 2002-10-09 -47.3% 2006-11-14     1670    True
2007-10-09 2009-03-09 -54.9% 2012-08-16     1224    True
2018-09-20 2018-12-24 -19.1% 2019-04-08      136   False
2020-02-19 2020-03-23 -33.7% 2020-08-10      120    True
2022-01-03 2022-10-12 -24.4% 2023-12-13      489    True
2025-02-19 2025-04-08 -18.7% 2025-06-26       88   False

seances en tension : 1315 sur 6709, soit 19.6 %
episodes majeurs : 4


In [5]:
import yfinance as yf

TAUX = RACINE / "data" / "raw" / "taux_sans_risque.csv"

if TAUX.exists():
    print(f"{TAUX.name} existe deja, collecte ignoree")
else:
    h = yf.Ticker("^IRX").history(period="max", auto_adjust=False)
    if h.empty:
        raise ValueError("aucune donnee pour ^IRX")
    serie = h["Close"].rename("taux_annuel_pct")
    serie.index = [str(d.date()) for d in serie.index]
    serie.index.name = "date"
    serie.to_frame().assign(symbole="^IRX").to_csv(TAUX, encoding="utf-8")
    print(len(serie), "lignes ecrites dans", TAUX.name)

taux = pd.read_csv(TAUX, index_col=0)["taux_annuel_pct"]
taux = taux.reindex(tout.index).ffill()

sans_risque = (1 + taux / 100) ** (1 / 252) - 1

print("couverture :", int(taux.notna().sum()), "sur", len(tout.index))
print("taux annuel : moyenne %.2f %% | min %.2f %% | max %.2f %%"
      % (taux.mean(), taux.min(), taux.max()))
print("jours a taux negatif :", int((taux < 0).sum()))

16655 lignes ecrites dans taux_sans_risque.csv
couverture : 6709 sur 6709
taux annuel : moyenne 1.92 % | min -0.10 % | max 6.22 %
jours a taux negatif : 7
